# 🏁 因果推断马拉松：0-1 评估营销活动 ROI (Causal Inference)

> **本次马拉松目标**：
> 在无法像 A/B 测试那样随机分组的情况下 (Observational Data)，如何准确评估一个策略的真实效果？
>
> **核心技能点**：
> 1.  **数据生成 (Simulation)**: 构造包含 "选择偏差" (Selection Bias) 的数据。
> 2.  **Naive 比较**: 直接对比两组均值，看看 "辛普森悖论" 是如何骗人的。
> 3.  **PSM (倾向性得分匹配)**: 使用 `LogisticRegression` 计算 Propensity Score，并进行配对。
> 4.  **平衡性检验 (Balance Check)**: 检查配对后的特征分布是否一致。
> 5.  **DoWhy 验证**: 使用 `dowhy` 库进行敏感性分析 (Placebo Test)，证明你的结论不是凑巧。

---

## 1. 模拟数据生成 (Data Generation)
**背景**：
我们上线了一个 "会员日大促" 活动 (`is_treated` = 1)。
但是，**并没有随机分组**。只有比较活跃、比较有钱的用户才更有可能参加这个活动。
如果你直接对比参加与不参加的人的购买力，肯定会**高估**活动的效果 (因为参加的人本来就更有钱)。

我们需要生成这种 "有偏" 的数据。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(1024)
n_samples = 5000

# 混杂因素 (Confounders)
# 用户的特征，这会同时影响 "是否参加活动" 和 "后续消费"
data = {
    'user_id': range(n_samples),
    'age': np.random.normal(35, 10, n_samples).astype(int),
    'income': np.random.normal(10000, 3000, n_samples), # 月收入
    'activity_level': np.random.uniform(0, 10, n_samples) # 活跃度 0-10
}
df = pd.DataFrame(data)

# 产生 Treatment (是否参加活动)
# 逻辑：收入高、活跃度高的人，更容易参加活动
prob_treated = 1 / (1 + np.exp(-(df['income']/5000 + df['activity_level']/2 - 5)))
df['is_treated'] = np.random.binomial(1, prob_treated)

# 产生 Outcome (活动后的消费金额)
# 真实因果效应 (ATE): 参加活动会让消费增加 500 元
# 但是，Outcome 也深受 income 和 activity_level 影响
true_causal_effect = 500
noise = np.random.normal(0, 200, n_samples)

df['spend'] = (
    200 + 
    0.05 * df['income'] + 
    100 * df['activity_level'] + 
    true_causal_effect * df['is_treated'] + 
    noise
)

print(f"Data Generated. Treatment Rate: {df['is_treated'].mean():.2%}")
df.head()

## 2. Naive Estimate (直接对比)
如果我们不懂因果推断，直接拿 `is_treated=1` 和 `is_treated=0` 的人对比，结果会是多少？
请计算两组的平均 `spend` 差值，并与真实的 500 元对比。

In [ ]:
# 🔍 你的代码：计算 Naive Difference
naive_diff = df[df['is_treated']==1]['spend'].mean() - df[df['is_treated']==0]['spend'].mean()
print(f"Naive Difference: {naive_diff:.2f}")
print(f"True Effect: {true_causal_effect}")
print(f"Bias: {naive_diff - true_causal_effect:.2f}")

## 3. PSM (倾向性得分匹配)
为了消除偏差，我们要找到一群 "如果不参加活动，特征和参加活动的人一模一样" 的替身。

**任务**：
1.  **训练模型**: 使用 `LogisticRegression` 预测 `is_treated`，特征为 `['income', 'activity_level', 'age']`。
2.  **获取分数**: 得到每个人的 Propensity Score。
3.  **匹配**: (简单版) 按照分数最接近的原则，给每个 Treated 用户找一个 Control 用户。

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

# 1. 计算 Propensity Score
# ps_model = LogisticRegression()
# X = ...
# y = ...
# df['ps_score'] = ...

# 2. 可视化 PS 分布 (检查重叠度 Common Support)
# sns.kdeplot(...)

# 3. 执行匹配 (Matching) - 这是一个难点，可以使用 NearestNeighbors
# treated = df[df['is_treated']==1]
# control = df[df['is_treated']==0]
# nn = NearestNeighbors(n_neighbors=1).fit(control[['ps_score']])
# distances, indices = nn.kneighbors(treated[['ps_score']])
# matched_control = control.iloc[indices.flatten()]


## 4. 评估匹配效果 (Balance Check & ATE Estimation)
匹配完了，一定要检查 **Trait Balance**。
匹配后的 Treated 组和 Control 组，在 `income` 和 `activity_level` 上应该没有显著差异了。

**任务**：
1.  对比匹配前后，特征均值的差异。
2.  **计算 ATE**: 匹配后的 Treated 组平均消费 - 匹配后的 Control 组平均消费。

In [ ]:
# 📊 你的代码：Balance Check

# 💰 你的代码：计算 PSM 调整后的 ATE


## 5. (进阶) 使用 DoWhy 自动化验证
手动写 PSM 很累？DoWhy 库可以一键搞定，并且自带 "安慰剂检验" (Placebo Test)。

**任务**：
1.  定义 Causal Model (Graph)。
2.  Identify Estimand。
3.  Estimate Effect (使用 `propensity_score_weighting` 或 `matching`)。
4.  Refute (验证): 如果我把 Outcome 换成随机数，DoWhy 应该告诉我 Effect 接近 0。

In [ ]:
import dowhy
from dowhy import CausalModel

# 🔮 你的代码：使用 DoWhy 全流程
# model = CausalModel(
#     data=df,
#     treatment='is_treated',
#     outcome='spend',
#     common_causes=['income', 'activity_level', 'age']
# )

# identified_estimand = model.identify_effect()
# estimate = model.estimate_effect(identified_estimand, method_name="backdoor.propensity_score_matching")
# print(estimate)

# refute = model.refute_estimate(identified_estimand, estimate, method_name="placebo_treatment_refuter")
# print(refute)
